# Hull-White One-Factor Model — Bond Pricing and Swaption Calibration
## QRE-44

Hull-White (1990) extends Vasicek by replacing the constant long-run mean $\theta$ with a time-dependent drift $\theta(t)$ chosen so that the model reproduces the **entire market OIS term structure exactly**. This one change turns a pedagogical model into the industry standard for XVA exposure simulation under collateralised derivative frameworks (EMIR, IFRS 13).

**What this notebook adds beyond QRE-48:**
</small>

| Topic | Covered in QRE-48 | New in QRE-44 |
|---|---|---|
| HW SDE and $\theta(t)$ derivation | Yes — full derivation | Only recap |
| Euler-Maruyama simulation | Yes | Cross-reference |
| Antithetic variates | Yes | Cross-reference |
| **Future ZCB price $P^{HW}(t,T)$** | No | **Yes — affine formula** |
| **European swaption pricing** | No | **Yes — Jamshidian** |
| **Calibrating $\kappa$ and $\sigma$ to vol surface** | No | **Yes** |
<small>

**Prerequisites:** [QRE-43 Vasicek](05_vasicek_model.ipynb) for the affine term structure and [QRE-48 MC Rate Paths](01_mc_rate_paths.ipynb) for the HW SDE and simulation mechanics.

---

**Regulatory relevance**

Hull-White calibrated to the ESTR OIS curve is the standard model for:
<small>

| Framework | Use |
|---|---|
| EMIR (648/2012) | Collateralised derivative MTM and exposure calculation |
| IFRS 13 | Level 2 fair value — CVA/FVA under OIS discounting |
| SA-CCR (CRR3 Art. 274) | Supervisory delta for interest rate options |
| CVA SA (CRR3 Art. 383) | Exposure simulation for regulatory CVA capital |
</small>


In [ ]:
from scipy.optimize import brentq, minimize
from scipy.stats import norm

from quant_risk.setup import base
from quant_risk.config import PROCESSED_DIR
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import HullWhiteProcess

np, pd, plt = base()

RNG_SEED = 42

---
## 1. Hull-White Recap and OIS Curve

### 1.1 The SDE (Recap — see QRE-48 for the full derivation)

$$dr(t) = \bigl(\theta(t) - \kappa\, r(t)\bigr)\,dt + \sigma\,dW(t)$$

The drift function:

$$\theta(t) = \frac{\partial f^M(0,t)}{\partial t} + \kappa\, f^M(0,t) + \frac{\sigma^2}{2\kappa}\bigl(1 - e^{-2\kappa t}\bigr)$$

ensures $\mathbb{E}^{\mathbb{Q}}[r(t)] = f^M(0,t)$ for all $t$ — the model mean-path tracks the OIS forward curve exactly.

### 1.2 What κ and σ Control

After fixing $\theta(t)$ from the OIS curve, two free parameters remain:

| Parameter | Effect on rates | Effect on swaptions |
|---|---|---|
| $\kappa$ | Speed of mean reversion; half-life $= \ln 2/\kappa$ | Controls vol term structure: high $\kappa$ → fast vol decay with maturity |
| $\sigma$ | Instantaneous rate vol (%/$\sqrt{\text{year}}$) | Controls the overall vol level |

These are calibrated to the **swaption volatility surface** (Section 4), not to ZCB prices (those are already matched by $\theta(t)$).


In [ ]:
# ── Load OIS curve ────────────────────────────────────────────────────────────
try:
    ois = OISCurve.from_processed(str(PROCESSED_DIR))
    print(ois.describe())
except FileNotFoundError:
    # Synthetic fallback: typical 2026 EUR inverted curve
    print("Processed OIS not found — using synthetic curve (run 02_bootstrapping_ois first)")
    data = pd.DataFrame(
        {"years":           [1/12, 2/12, 3/12, 6/12, 9/12, 1.0,  2.0,  3.0,  5.0,  10.0, 15.0],
         "zero_rate_pct":   [2.63, 2.61, 2.58, 2.48, 2.38, 2.30, 2.20, 2.15, 2.20,  2.40, 2.50],
         "discount_factor": [np.exp(-0.01 * r * t)
                              for r, t in zip(
                                  [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
                                  [1/12,2/12,3/12,6/12,9/12,1,2,3,5,10,15]
                              )],
         "valuation_date":  ["2026-03-24"] * 11},
        index=["1M","2M","3M","6M","9M","12M","2Y","3Y","5Y","10Y","10Y+"])
    data.index.name = "maturity"
    ois = OISCurve(data)
    print(ois.describe())

# Anchor: r(0) = 1M→2M OIS forward (avoids integer-day arithmetic at t=0)
r0 = ois.forward_rate(1/12, 2/12)
print(f"\nr(0) = {r0:.4f}%  (1M→2M OIS forward rate)")

# Show OIS forward curve — this is what θ(t) will reproduce in expectation
t_fwd  = np.array([k/12 for k in range(1, 25)] + list(range(3, 16)))
f_fwd  = np.array([ois.forward_rate(t, t + 1/12) for t in t_fwd])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_fwd, f_fwd, 'o-', color='steelblue', ms=3, lw=1.5,
        label='OIS instantaneous forward $f^M(0,t)$')
ax.axhline(r0, lw=0.9, ls='--', color='gray', alpha=0.7,
           label=f'r(0) = {r0:.2f}%')
ax.set_xlabel('t (years)')
ax.set_ylabel('Forward rate (%)')
ax.set_title('OIS forward curve — Hull-White calibration target')
ax.legend()
plt.tight_layout()
plt.show()


---
## 2. Future Bond Prices Under Hull-White

### 2.1 The Affine Formula

Hull-White retains the affine term structure: given the short rate $r(t)$ at some future time $t$, the ZCB price for maturity $T > t$ is:

$$\boxed{P^{HW}(t, T;\, r(t)) = \frac{P^M(0,T)}{P^M(0,t)} \cdot \exp\!\left[-B(T-t)\cdot\bigl(r(t) - f^M(0,t)\bigr) - \frac{\sigma^2}{4\kappa}\,B(T-t)^2\,\bigl(1 - e^{-2\kappa t}\bigr)\right]}$$

where $B(\tau) = (1 - e^{-\kappa\tau})/\kappa$ as in Vasicek.

### 2.2 Three-Term Anatomy

The three factors express how the model bond price departs from the pure market ZCB:
<small>

| Factor | Expression | Interpretation |
|---|---|---|
| Market ratio | $P^M(0,T)/P^M(0,t)$ | The market forward ZCB from $t$ to $T$; the HW model reproduces it on average |
| Rate deviation | $\exp[-B(T-t)(r(t)-f^M(0,t))]$ | If $r(t) > f^M(0,t)$, the simulated rate is above the forward rate → bonds are cheaper than the market expects |
| Variance adjustment | $\exp[-\frac{\sigma^2}{4\kappa}B(T-t)^2(1-e^{-2\kappa t})]$ | Jensen's inequality correction: $E[e^{-\int r}] < e^{-\int E[r]}$; small at $t=0$, grows over time |

</small>


### 2.3 Consistency Check at $t = 0$

At $t=0$: $P^M(0,0) = 1$, $f^M(0,0) = r(0)$, and $(1 - e^{-2\kappa \cdot 0}) = 0$. Therefore:

$$P^{HW}(0,T;\, r(0)) = P^M(0,T) \cdot \exp[-B(T)\cdot 0 - 0] = P^M(0,T) \checkmark$$

The model reproduces today's OIS discount factors exactly at time zero — the defining property of a **no-arbitrage** model.

### 2.4 Why This Formula Matters for XVA

In the XVA exposure profile, the MTM of a swap at simulation time $t$ requires pricing a sequence of future ZCBs $\{P^{HW}(t, T_i; r(t))\}$ along each simulated path. This formula does that in closed form — no nested Monte Carlo needed.


In [ ]:
# ── Hull-White future bond price P^HW(t, T; r) ──────────────────────────────

def hw_B(tau: np.ndarray, kappa: float) -> np.ndarray:
    """B(τ) = (1 - e^{-κτ}) / κ.  Handles τ=0 by returning 0."""
    return np.where(tau > 0, (1 - np.exp(-kappa * tau)) / kappa, 0.0)

def hw_bond_price(t: float, T: np.ndarray, r_t: np.ndarray,
                  kappa: float, sigma: float, curve: OISCurve) -> np.ndarray:
    """
    P^HW(t, T; r(t)) for a vector of maturities T and paths r_t.

    Parameters
    ----------
    t : float
        Current (simulation) time in years.
    T : np.ndarray
        Maturity dates in years (shape (n_maturities,)).
    r_t : np.ndarray
        Short rate at time t along each path (shape (n_paths,)).
    kappa, sigma : float
        Hull-White parameters.
    curve : OISCurve
        Market term structure.

    Returns
    -------
    np.ndarray
        Shape (n_paths, n_maturities).
    """
    tau   = T - t                                  # time to maturity τ = T - t
    B_tau = hw_B(tau, kappa)                       # B(τ), shape (n_maturities,)

    # Market discount factors and forward rate at time t
    # P^M(0,T)/P^M(0,t) = the market forward discount factor
    P_0T  = np.array([curve.discount(Ti) for Ti in T])
    P_0t  = curve.discount(t) if t > 1e-6 else 1.0

    # Instantaneous forward rate f^M(0,t): use a 1M window to match HullWhiteProcess._theta()
    dt_fd = 1.0 / 12.0
    t_lo  = max(t - dt_fd, dt_fd)
    f_0t  = curve.forward_rate(t_lo, t_lo + dt_fd)

    # Variance adjustment: (σ_dec)²/(4κ)·B²·(1-e^{-2κt}); σ is in %/√yr → /100
    sigma_d = sigma / 100
    var_adj = (sigma_d**2 / (4 * kappa)) * B_tau**2 * (1 - np.exp(-2 * kappa * t))

    # Rate deviation term: (r(t) - f^M(0,t)) in percent → divide by 100
    rate_dev = (r_t[:, None] - f_0t) / 100 * B_tau[None, :]  # (n_paths, n_maturities)

    return (P_0T / P_0t)[None, :] * np.exp(-rate_dev - var_adj[None, :])

# ── Verification 1: P^HW(0, T; r0) must equal P^M(0, T) ─────────────────────
kappa_test, sigma_test = 0.10, 0.50
test_maturities = np.array([1.0, 2.0, 5.0, 10.0])

P_hw_t0 = hw_bond_price(t=1e-9, T=test_maturities,
                          r_t=np.array([r0]),
                          kappa=kappa_test, sigma=sigma_test, curve=ois)
P_ois   = np.array([ois.discount(T) for T in test_maturities])

print("Consistency check at t≈0: P^HW(0,T; r0) vs P^M(0,T)")
print(f"  {'T':>6}  {'P^HW':>12}  {'P^OIS':>12}  {'Diff (bps)':>12}")
for T, Phw, Pois in zip(test_maturities, P_hw_t0[0], P_ois):
    print(f"  {T:>6.1f}Y  {Phw:>12.8f}  {Pois:>12.8f}  {(Phw-Pois)*1e4:>+12.4f}")

print("\nMax abs error:", abs(P_hw_t0[0] - P_ois).max())

In [ ]:
# ── Verification 2: cross-section of bond prices at a future time t=2Y ───────
# At t=2Y with rates above and below f^M(0,2Y), show how bond prices shift.

t_future = 2.0
f_future = ois.forward_rate(t_future - 1/12, t_future + 1/12)
future_maturities = np.array([3.0, 4.0, 5.0, 7.0, 10.0])

# Three rate scenarios at t=2Y
r_scenarios = np.array([f_future - 1.0, f_future, f_future + 1.0])  # ±100 bps

P_scenarios = hw_bond_price(t=t_future, T=future_maturities,
                              r_t=r_scenarios,
                              kappa=kappa_test, sigma=sigma_test, curve=ois)

fig, ax = plt.subplots(figsize=(9, 4))
labels_scen = [f'r(2Y) = {r:.2f}%  (f^M(0,2Y){r-f_future:+.0f}bps×100)'
               for r in r_scenarios]
colors_scen = ['steelblue', 'darkorange', 'firebrick']

for i, (label, color) in enumerate(zip(labels_scen, colors_scen)):
    ax.plot(future_maturities, P_scenarios[i], 'o-', lw=1.5, ms=5,
            color=color, label=label)

# Overlay the market forward ZCBs for reference
P_market_fwd = np.array([ois.discount(T) / ois.discount(t_future)
                          for T in future_maturities])
ax.plot(future_maturities, P_market_fwd, 'k--', lw=1.2, alpha=0.6,
        label=f'Market forward $P^M(0,T)/P^M(0,{t_future:.0f}Y)$')

ax.set_xlabel('Maturity T (years)')
ax.set_ylabel(f'$P^{{HW}}({t_future:.0f}Y, T;\, r)$')
ax.set_title(f'HW bond prices at t={t_future:.0f}Y under three rate scenarios\n'
             f'κ={kappa_test}, σ={sigma_test}%/√yr')
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

print(f"f^M(0, {t_future}Y) = {f_future:.4f}%  (at this rate, HW price = market forward ZCB)")


---
## 3. European Swaption Pricing — ZCB Options and Jamshidian Decomposition

### 3.1 ZCB Call Options Under Hull-White

For a European call option on a ZCB $P(T_0, T_1)$ with strike $X$ and option maturity $T_0$:

$$\boxed{C(0,\, T_0,\, T_1,\, X) = P^M(0,T_1)\,\mathcal{N}(h_1) - X\cdot P^M(0,T_0)\,\mathcal{N}(h_2)}$$

$$h_{1,2} = \frac{\ln\!\left[P^M(0,T_1)\big/\bigl(X\cdot P^M(0,T_0)\bigr)\right]}{\sigma_P} \pm \frac{\sigma_P}{2}$$

$$\sigma_P(T_0,T_1) = \frac{\sigma}{\kappa}\bigl(1 - e^{-\kappa(T_1-T_0)}\bigr) \sqrt{\frac{1 - e^{-2\kappa T_0}}{2\kappa}}$$

$\sigma_P$ is the **total volatility of the log ZCB price** between 0 and $T_0$. It equals $\sigma \cdot B(T_1-T_0) \cdot v(T_0)$ where $v(T_0) = \sqrt{(1-e^{-2\kappa T_0})/(2\kappa)}$ is the standard deviation of $r(T_0)$. Qualitatively: higher $\sigma$, lower $\kappa$ (slower mean reversion), or longer $T_0$ all increase $\sigma_P$.

By put-call parity:
$$P(0, T_0, T_1, X) = C - P^M(0,T_1) + X\cdot P^M(0,T_0)$$

### 3.2 Payer Swaption — Jamshidian Decomposition (1989)

A **payer swaption** gives the holder the right at $T_0$ to enter a swap paying fixed rate $K$ and receiving floating. It can be decomposed into **put options on ZCBs**.

At $T_0$, the payer swaption is exercised when the swap rate $S(T_0) > K$, which is equivalent to the **coupon bond price** being below par:

$$\sum_{i=1}^n c_i \cdot P(T_0, T_i;\, r(T_0)) < 1$$

where $c_i = K\delta_i$ for $i < n$ and $c_n = 1 + K\delta_n$ ($\delta_i$ = year fraction).

**Jamshidian's result**: in any one-factor model where $P(T_0, T_i)$ is monotone in $r(T_0)$, the payer swaption value equals:

$$V_{\text{payer}} = \sum_{i=1}^n c_i \cdot \text{Put}\bigl(0,\, T_0,\, T_i,\, X^*_i\bigr)$$

where $X^*_i = P^{HW}(T_0, T_i;\, r^*)$ and $r^*$ is the unique rate solving:

$$\sum_{i=1}^n c_i \cdot P^{HW}(T_0, T_i;\, r^*) = 1$$

**Why this works**: at exactly $r^*$, the swap is at par. For $r > r^*$ (swaption exercised), each ZCB $P(T_0,T_i) < X^*_i$ — so each ZCB put is in-the-money. The cashflow weights $c_i$ ensure the put values sum to the swaption payoff.

### 3.3 Normal Implied Volatility

Swaption prices are conventionally quoted in **normal (Bachelier) implied vol** $\sigma_N$ in bps/year. For an at-the-money swaption with annuity $A = \sum_i \delta_i P^M(0,T_i)$ and expiry $T_0$:

$$V_{\text{payer}}^{\text{ATM}} = A \cdot \frac{\sigma_N \sqrt{T_0}}{\sqrt{2\pi}}$$

$$\Rightarrow \quad \sigma_N = V_{\text{payer}} \cdot \frac{\sqrt{2\pi}}{A \sqrt{T_0}} \cdot 10{,}000 \quad \text{(bps/year)}$$

This is the metric used to compare model prices to market quotes.


In [ ]:
# ── Hull-White ZCB option pricing ────────────────────────────────────────────

def hw_sigma_P(T0: float, T1: float, kappa: float, sigma: float) -> float:
    """
    Total log-price vol σ_P for a ZCB maturing at T1, option expiring at T0.
    σ_P = σ_dec · B(T1-T0) · √[(1-e^{-2κT0})/(2κ)]
    sigma is in %/√yr; divide by 100 to get σ_dec for a dimensionless σ_P.
    """
    B_tau = (1 - np.exp(-kappa * (T1 - T0))) / kappa
    v_T0  = np.sqrt((1 - np.exp(-2 * kappa * T0)) / (2 * kappa))
    return (sigma / 100) * B_tau * v_T0

def hw_zcb_call(T0: float, T1: float, X: float,
                kappa: float, sigma: float, curve: OISCurve) -> float:
    """European call option on P(T0, T1) with strike X, priced at time 0."""
    sig_P = hw_sigma_P(T0, T1, kappa, sigma)
    if sig_P < 1e-12:
        # Zero volatility: option is worth max(P^M(0,T1)/P^M(0,T0) - X, 0) · P^M(0,T0)
        fwd_P = curve.discount(T1) / curve.discount(T0)
        return max(fwd_P - X, 0.0) * curve.discount(T0)
    P_0T0 = curve.discount(T0)
    P_0T1 = curve.discount(T1)
    h1 = np.log(P_0T1 / (X * P_0T0)) / sig_P + sig_P / 2
    h2 = h1 - sig_P
    return P_0T1 * norm.cdf(h1) - X * P_0T0 * norm.cdf(h2)

def hw_zcb_put(T0: float, T1: float, X: float,
               kappa: float, sigma: float, curve: OISCurve) -> float:
    """European put option on P(T0, T1) with strike X, priced at time 0."""
    # Put-call parity: Put = Call - P^M(0,T1) + X·P^M(0,T0)
    call  = hw_zcb_call(T0, T1, X, kappa, sigma, curve)
    P_0T0 = curve.discount(T0)
    P_0T1 = curve.discount(T1)
    return call - P_0T1 + X * P_0T0

# ── Visualise σ_P as a function of (T0, T1-T0) ──────────────────────────────
T0_grid    = np.linspace(0.5, 10, 30)
tenor_grid = np.array([1.0, 2.0, 5.0, 10.0])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))

# Left: σ_P surface
ax = ax0
for tenor, color in zip(tenor_grid, ['steelblue','darkorange','forestgreen','firebrick']):
    sig_P_vals = [hw_sigma_P(T0, T0 + tenor, kappa_test, sigma_test)
                  for T0 in T0_grid]
    ax.plot(T0_grid, np.array(sig_P_vals) * 100,
            lw=1.5, color=color, label=f'tenor={tenor:.0f}Y')
ax.set_xlabel('Option expiry $T_0$ (years)')
ax.set_ylabel(r'$\sigma_P(T_0, T_1)$ (%)')
ax.set_title(f'Total ZCB price vol $\sigma_P$\n(κ={kappa_test}, σ={sigma_test}%/√yr)')
ax.legend(fontsize=7)

# Right: effect of κ on σ_P for a fixed 5Y option on a 10Y ZCB
ax = ax1
T0_fixed, T1_fixed = 5.0, 10.0
sigma_fixed = 0.50
for kappa_val, color in [(0.03,'steelblue'),(0.10,'darkorange'),(0.30,'firebrick')]:
    sig_P_kappa = [hw_sigma_P(T0, T0 + (T1_fixed - T0_fixed), kappa_val, sigma_fixed)
                   for T0 in T0_grid]
    hl = np.log(2) / kappa_val
    ax.plot(T0_grid, np.array(sig_P_kappa) * 100,
            lw=1.5, color=color,
            label=f'κ={kappa_val:.2f}  (HL {hl:.1f}y)')
ax.set_xlabel('Option expiry $T_0$ (years)')
ax.set_ylabel(r'$\sigma_P$ (%)')
ax.set_title(f'Effect of κ on ZCB vol\n(σ={sigma_fixed}%, tenor={T1_fixed-T0_fixed:.0f}Y)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print("Insight: higher κ → faster mean reversion → shorter effective rate vol horizon → lower σ_P.")
print("This is how κ controls the term structure of swaption vols (see Section 4).")

In [ ]:
# ── Jamshidian payer swaption pricer ─────────────────────────────────────────

def hw_jamshidian_payer(T0: float, payment_dates: np.ndarray,
                         year_fracs: np.ndarray, K: float,
                         kappa: float, sigma: float, curve: OISCurve) -> float:
    """
    Payer swaption price via Jamshidian decomposition.

    Parameters
    ----------
    T0 : float
        Option expiry = swap start date (years).
    payment_dates : np.ndarray
        Swap payment dates T_1, ..., T_n in years.
    year_fracs : np.ndarray
        Day-count fractions δ_i for each payment period.
    K : float
        Fixed coupon rate in percent (e.g. 2.5 for 2.5%).
    kappa, sigma : float
        Hull-White parameters.
    curve : OISCurve
        Market term structure.

    Returns
    -------
    float
        Payer swaption price per unit notional.
    """
    K_dec = K / 100.0   # convert to decimal for cashflow calculation

    # Cashflows: K·δ_i for i<n, 1 + K·δ_n at maturity
    cashflows = K_dec * year_fracs
    cashflows[-1] += 1.0

    # Helper: P^HW(T0, Ti; r) using the affine formula evaluated at T0
    # A^HW(T0, Ti) = P^M(0,Ti)/P^M(0,T0) · exp[f^M(0,T0)·B(Ti-T0) - var_adj]
    dt_fd = 1.0 / 12.0
    f_T0  = curve.forward_rate(T0 - dt_fd, T0 + dt_fd)
    P_0T0 = curve.discount(T0)

    tau_i = payment_dates - T0
    B_i   = hw_B(tau_i, kappa)
    P_0Ti = np.array([curve.discount(Ti) for Ti in payment_dates])
    sigma_d   = sigma / 100
    var_adj_i = (sigma_d**2 / (4 * kappa)) * B_i**2 * (1 - np.exp(-2 * kappa * T0))

    # A^HW(T0, Ti) — pre-computed factor independent of r
    log_A = np.log(P_0Ti / P_0T0) + f_T0 * B_i - var_adj_i

    def bond_value(r: float) -> float:
        """Coupon bond value at T0 as a function of the short rate r."""
        log_P = log_A - B_i * r
        return (cashflows * np.exp(log_P)).sum()

    # Find r* such that bond_value(r*) = 1 (par) using bracketed root search.
    # bond_value is monotone decreasing in r, so we bracket with wide limits.
    try:
        r_star = brentq(lambda r: bond_value(r) - 1.0, a=-5.0, b=30.0, xtol=1e-10)
    except ValueError:
        # No root — option is deep in- or out-of-the-money
        if bond_value(-5.0) < 1.0:
            return 0.0   # deep OTM payer (rates too low for any exercise)
        return float('nan')

    # X*_i = P^HW(T0, Ti; r*)
    X_star = np.exp(log_A - B_i * r_star)

    # Sum of weighted ZCB puts
    payer_price = sum(
        ci * hw_zcb_put(T0, Ti, Xi, kappa, sigma, curve)
        for ci, Ti, Xi in zip(cashflows, payment_dates, X_star)
    )
    return payer_price

def atm_swap_rate(T0: float, payment_dates: np.ndarray,
                   year_fracs: np.ndarray, curve: OISCurve) -> float:
    """ATM fixed rate for a swap starting T0 with given payment schedule (in percent)."""
    P_0T0 = curve.discount(T0)
    P_end  = curve.discount(payment_dates[-1])
    annuity = (year_fracs * np.array([curve.discount(Ti)
                                       for Ti in payment_dates])).sum()
    return (P_0T0 - P_end) / annuity * 100   # in percent

def bachelier_normal_vol(price: float, T0: float,
                          annuity: float, forward_rate: float,
                          K: float) -> float:
    """
    Bachelier (normal) implied vol in bps/year from a swaption price.
    For ATM (forward_rate ≈ K), reduces to price / (annuity · √T0/√2π) × 10000.
    """
    if abs(forward_rate - K) < 1e-8:
        # ATM formula: V = A · σ_N · √T0 / √(2π)
        return price / annuity / np.sqrt(T0 / (2 * np.pi)) * 10000  # bps/yr
    # General Bachelier: solve numerically (omitted for brevity)
    return price / annuity / np.sqrt(T0 / (2 * np.pi)) * 10000

# ── Price a representative swaption and compute normal vol ───────────────────
# Example: 2Y×5Y payer swaption (option expiry 2Y, swap tenor 5Y)
T0_ex     = 2.0
T_n_ex    = 7.0
dates_ex  = np.arange(T0_ex + 0.5, T_n_ex + 0.5, 0.5)  # semi-annual: 2.5, 3.0, ..., 7.0
yf_ex     = np.full(len(dates_ex), 0.5)                  # 30/360: δ = 0.5

F_ex      = atm_swap_rate(T0_ex, dates_ex, yf_ex, ois)
annuity_ex = (yf_ex * np.array([ois.discount(Ti) for Ti in dates_ex])).sum()

print(f"2Y×5Y ATM payer swaption")
print(f"  ATM swap rate:   {F_ex:.4f}%")
print(f"  Annuity:         {annuity_ex:.6f}")
print(f"  Payment dates:   {dates_ex}")

# Price at different (κ, σ) combinations
for kappa_e, sigma_e in [(0.05, 0.40), (0.10, 0.50), (0.20, 0.60)]:
    price = hw_jamshidian_payer(T0_ex, dates_ex, yf_ex, F_ex,
                                  kappa_e, sigma_e, ois)
    sigma_N = bachelier_normal_vol(price, T0_ex, annuity_ex, F_ex/100, F_ex/100)
    print(f"  κ={kappa_e:.2f} σ={sigma_e:.2f}%:  price={price:.6f}  σ_N={sigma_N:.1f} bps/yr")

---
## 4. Calibrating κ and σ to the ATM Swaption Surface

### 4.1 The Vol Surface Structure

The ATM swaption surface is indexed by **(expiry, tenor)**. Hull-White, being a 1-factor Gaussian model, cannot reproduce an arbitrary 2D surface — but it can fit the **level and term structure** of vol reasonably well for a calibration that targets the key XVA-relevant region (2Y–10Y expiry, 5Y–10Y tenor).

### 4.2 Normal Vol Convention

EUR swaptions are quoted in **normal (Bachelier) implied vol** in bps/year because EURIBOR/ESTR went negative 2014–2022. The relationship between $\kappa$, $\sigma$, and normal vol is approximately:

$$\sigma_N^{(T_0, \text{tenor})} \approx \frac{V_{\text{Jamshidian}}}{A \sqrt{T_0/(2\pi)}} \times 10{,}000 \quad \text{bps/yr}$$

Qualitatively:
- **$\sigma$ scales vol level** — doubling $\sigma$ ≈ doubles all normal vols
- **$\kappa$ controls the term structure slope** — higher $\kappa$ suppresses long-expiry   vols (fast mean reversion dampens long-horizon uncertainty)

### 4.3 Calibration Objective

Given market ATM normal vols $\{\sigma_N^{(i)}\}$, find $(\kappa, \sigma)$ solving:

$$\min_{\kappa,\, \sigma} \sum_i w_i \bigl(\sigma_N^{\text{model}}(\kappa,\sigma; \text{expiry}_i, \text{tenor}_i) - \sigma_N^{(i)}\bigr)^2$$

with weights $w_i = T_0^{(i)} \cdot \text{tenor}_i$ (longer expiry × longer tenor = higher XVA weight).

We use a synthetic ATM surface calibrated to typical 2026 EUR conditions at a ~2.5% rate level.


In [ ]:
# ── Synthetic EUR ATM normal swaption vol surface ────────────────────────────
# Typical EUR ATM normal vols (bps/year) for a ~2.5% rate environment (2026).
# Sources: ECB/Bloomberg EUR swaption market conventions.
# These are approximate; replace with live data from a vol provider in production.

expiries = np.array([1.0, 2.0, 5.0, 10.0])   # option expiries in years
tenors   = np.array([1.0, 2.0, 5.0, 10.0])    # swap tenors in years

# ATM normal implied vols in bps/year
# Shape: (len(expiries), len(tenors)) = (4, 4)
market_vols_bps = np.array([
    [38, 42, 48, 52],   # 1Y expiry
    [42, 46, 52, 56],   # 2Y expiry
    [48, 52, 57, 60],   # 5Y expiry
    [50, 55, 58, 61],   # 10Y expiry
], dtype=float)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))

# Left: vol surface as a heatmap
ax = ax0
im = ax.imshow(market_vols_bps, cmap='RdYlGn_r', aspect='auto',
               vmin=35, vmax=65)
ax.set_xticks(range(len(tenors)))
ax.set_xticklabels([f'{t:.0f}Y' for t in tenors])
ax.set_yticks(range(len(expiries)))
ax.set_yticklabels([f'{e:.0f}Y' for e in expiries])
ax.set_xlabel('Swap tenor')
ax.set_ylabel('Option expiry')
ax.set_title('EUR ATM normal swaption vol surface\n(bps/year, synthetic 2026)')
for i in range(len(expiries)):
    for j in range(len(tenors)):
        ax.text(j, i, f'{market_vols_bps[i,j]:.0f}', ha='center', va='center',
                fontsize=9, color='black')
plt.colorbar(im, ax=ax, label='Normal vol (bps/yr)')

# Right: vol by expiry for different tenors (term structure of vol)
ax = ax1
for k, tenor in enumerate(tenors):
    ax.plot(expiries, market_vols_bps[:, k], 'o-', lw=1.5, ms=5,
            label=f'tenor={tenor:.0f}Y')
ax.set_xlabel('Option expiry (years)')
ax.set_ylabel('ATM normal vol (bps/year)')
ax.set_title('Vol term structure by swap tenor')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print("Observation: vol increases with both expiry and tenor.")
print("HW can fit the level (via σ) and the expiry slope (via κ),")
print("but not independent curvature across the 2D surface — a 1-factor limitation.")


In [ ]:
# ── Calibration objective and optimisation ────────────────────────────────────

# Build the calibration grid: all (expiry, tenor) pairs
cal_expiries, cal_tenors, cal_target_vols = [], [], []
cal_weights = []

for i, exp in enumerate(expiries):
    for j, tenor in enumerate(tenors):
        payment_dates = np.arange(exp + 0.5, exp + tenor + 0.5, 0.5)
        if len(payment_dates) == 0:
            continue
        yf = np.full(len(payment_dates), 0.5)
        cal_expiries.append(exp)
        cal_tenors.append(tenor)
        cal_target_vols.append(market_vols_bps[i, j])
        cal_weights.append(exp * tenor)     # weight by expiry × tenor

cal_weights = np.array(cal_weights) / sum(cal_weights)

def model_normal_vol(exp: float, tenor: float,
                      kappa: float, sigma: float) -> float:
    """Compute HW normal implied vol for one swaption."""
    payment_dates = np.arange(exp + 0.5, exp + tenor + 0.5, 0.5)
    yf = np.full(len(payment_dates), 0.5)
    F_atm    = atm_swap_rate(exp, payment_dates, yf, ois)
    annuity  = (yf * np.array([ois.discount(Ti) for Ti in payment_dates])).sum()
    price    = hw_jamshidian_payer(exp, payment_dates, yf, F_atm, kappa, sigma, ois)
    return bachelier_normal_vol(price, exp, annuity, F_atm/100, F_atm/100)

def calibration_objective(params: np.ndarray) -> float:
    kappa_c, sigma_c = params
    if kappa_c <= 0 or sigma_c <= 0:
        return 1e10
    sse = 0.0
    for exp, tenor, target_vol, w in zip(
            cal_expiries, cal_tenors, cal_target_vols, cal_weights):
        try:
            mod_vol = model_normal_vol(exp, tenor, kappa_c, sigma_c)
            sse += w * (mod_vol - target_vol) ** 2
        except Exception:
            sse += w * 1000.0
    return sse

# Multi-start optimisation: explore the (κ, σ) space to avoid local minima
print("Running calibration... (may take ~30 seconds)")
x0_list = [(0.05, 0.35), (0.10, 0.50), (0.20, 0.65), (0.30, 0.80)]
best = None

for x0 in x0_list:
    res = minimize(calibration_objective, x0=np.array(x0),
                   method='Nelder-Mead',
                   options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-8})
    if best is None or res.fun < best.fun:
        best = res

kappa_cal, sigma_cal = best.x
print(f"\nCalibrated parameters:")
print(f"  κ = {kappa_cal:.4f}  (half-life {np.log(2)/kappa_cal:.1f}y)")
print(f"  σ = {sigma_cal:.4f}%/√yr")

# Residuals table
print(f"\n{'Expiry':>6}  {'Tenor':>5}  {'Market':>8}  {'Model':>8}  {'Error':>8}  {'Weight':>8}")
for exp, tenor, target, w in zip(cal_expiries, cal_tenors, cal_target_vols, cal_weights):
    mod = model_normal_vol(exp, tenor, kappa_cal, sigma_cal)
    print(f"  {exp:>5.1f}Y  {tenor:>4.0f}Y  {target:>8.1f}  {mod:>8.1f}  "
          f"{mod-target:>+8.1f}  {w:>8.4f}")

In [ ]:
# ── κ–σ landscape: contour plot of calibration RMSE ─────────────────────────
# This shows why (κ, σ) calibration can have a shallow valley — there are many
# parameter pairs with similar vol level but different term structures.

print("Computing κ-σ landscape... (may take ~60 seconds)")
kappa_grid = np.linspace(0.02, 0.40, 15)
sigma_grid = np.linspace(0.20, 1.00, 15)

landscape = np.zeros((len(kappa_grid), len(sigma_grid)))
for i, k in enumerate(kappa_grid):
    for j, s in enumerate(sigma_grid):
        landscape[i, j] = np.sqrt(calibration_objective(np.array([k, s])))

fig, ax = plt.subplots(figsize=(8, 5))
K, S = np.meshgrid(sigma_grid, kappa_grid)
cs = ax.contourf(K, S, landscape, levels=20, cmap='viridis_r')
ax.contour(K, S, landscape, levels=10, colors='white', linewidths=0.5, alpha=0.4)
ax.plot(sigma_cal, kappa_cal, 'r*', ms=14, label=f'Optimum: κ={kappa_cal:.3f}, σ={sigma_cal:.3f}%')
ax.set_xlabel('σ (%/√yr)')
ax.set_ylabel('κ (1/year)')
ax.set_title('Calibration RMSE landscape (bps/yr)')
plt.colorbar(cs, ax=ax, label='RMSE (bps/yr)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nInsight: the landscape shows a valley — many (κ,σ) pairs give similar overall RMSE.")
print("High κ + high σ and low κ + low σ can both match the vol level.")
print("The term structure of vol (how quickly vol decays with expiry) differentiates them.")


---
## 5. Simulation with Calibrated Parameters

With $\theta(t)$ calibrated to the OIS forward curve (built into `HullWhiteProcess`) and $(\kappa, \sigma)$ now calibrated to the swaption surface, we have a fully specified model.

**Two key properties to verify:**

1. $\mathbb{E}[r(t)] \approx f^M(0,t)$ — the simulated mean path tracks the OIS forward curve
2. The simulated ZCB prices $P^{HW}(t, T_i; r(t))$ averaged across paths match $P^M(0,T_i)$

Property 1 follows from the $\theta(t)$ calibration (see QRE-48 for the full proof). Property 2 follows from the law of iterated expectations applied to the bond pricing formula.


In [ ]:
# ── Simulate HW paths with calibrated (κ, σ) ─────────────────────────────────
hw_proc = HullWhiteProcess(curve=ois, kappa=kappa_cal, sigma=sigma_cal)
print(hw_proc.describe())

T_sim   = 10.0
n_steps = 120    # monthly steps — standard for XVA
n_paths = 2000
t_grid  = np.linspace(0, T_sim, n_steps + 1)

paths = hw_proc.simulate(
    x0        = r0,
    T         = T_sim,
    n_steps   = n_steps,
    n_paths   = n_paths,
    antithetic= True,
    seed      = RNG_SEED,
)

# ── Property 1: E[r(t)] vs f^M(0,t) ─────────────────────────────────────────
fwd_on_grid = np.array([ois.forward_rate(max(t - 1/12, 1/12), max(t, 2/12))
                         for t in t_grid])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.5))

ax = ax0
for i in range(40):
    ax.plot(t_grid, paths[i], lw=0.3, alpha=0.25, color='steelblue')
q5,  q95 = np.percentile(paths, [5, 95], axis=0)
q25, q75 = np.percentile(paths, [25, 75], axis=0)
ax.fill_between(t_grid, q5, q95, color='steelblue', alpha=0.10, label='5th–95th pctile')
ax.fill_between(t_grid, q25, q75, color='steelblue', alpha=0.20, label='25th–75th pctile')
ax.plot(t_grid, paths.mean(axis=0), '-',  color='darkorange', lw=1.5, label='Simulated E[r(t)]')
ax.plot(t_grid, fwd_on_grid,        '--', color='black',      lw=1.5, label='OIS forward $f^M(0,t)$')
ax.set_xlabel('t (years)')
ax.set_ylabel('r(t) (%)')
ax.set_title(f'HW fan chart — κ={kappa_cal:.3f}, σ={sigma_cal:.3f}%/√yr')
ax.legend(fontsize=7)

# Property 2: mean ZCB price across paths vs OIS discount factors
ax = ax1
verify_maturities = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
P_ois_verify = np.array([ois.discount(T) for T in verify_maturities])

# At t=2Y, average P^HW(2Y, T; r(2Y)) across paths — should equal P^M(0,T)
t_check = 2.0
t_idx   = int(round(t_check * n_steps / T_sim))
r_at_t  = paths[:, t_idx]

P_hw_avg = np.array([
    hw_bond_price(t_check, np.array([T]), r_at_t, kappa_cal, sigma_cal, ois).mean()
    for T in verify_maturities
])

# Discount back to t=0: E[P^HW(t,T)] × P^M(0,t) should ≈ P^M(0,T)
P_discounted = P_hw_avg * ois.discount(t_check)

ax.plot(verify_maturities, P_ois_verify,   'o-', color='black',     lw=1.5, ms=6,
        label='$P^M(0,T)$ (OIS market)')
ax.plot(verify_maturities, P_discounted,   's--', color='steelblue', lw=1.5, ms=6,
        label=f'$P^M(0,t) \cdot E[P^{{HW}}(t,T)]$  $t={t_check:.0f}$Y')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Discount factor')
ax.set_title(f'No-arbitrage check: $P^M(0,t)·E[P^{{HW}}(t,T;r(t))] \approx P^M(0,T)$')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Numerical check
print(f"No-arbitrage verification at t={t_check}Y:")
print(f"  {'T':>5}  {'P^M(0,T)':>12}  {'E[P^HW]×P^M(0,t)':>18}  {'Error (bps)':>12}")
for T, Pois, Phw_disc in zip(verify_maturities, P_ois_verify, P_discounted):
    print(f"  {T:>5.1f}Y  {Pois:>12.6f}  {Phw_disc:>18.6f}  {(Phw_disc-Pois)*1e4:>+12.2f}")

---
## 6. Hull-White vs Vasicek — Summary and When to Use Each

<small>

| Property | Vasicek | Hull-White |
|---|---|---|
| Fits OIS term structure exactly | No — calibration residuals at belly | Yes — by construction via $\theta(t)$ |
| Free parameters | $\kappa$, $\theta$, $\sigma$ (3) | $\kappa$, $\sigma$ (2) — $\theta(t)$ from curve |
| Calibrated to | OIS zero yields (imperfect fit) | Swaption vols; OIS exact by design |
| Rates can go negative | Yes | Yes |
| Analytical ZCB at $t=0$ | $P^{\text{Vasicek}}(0,T) \neq P^M(0,T)$ in general | $P^{HW}(0,T) = P^M(0,T)$ always |
| Analytical ZCB at $t > 0$ | Yes (but doesn't use market data) | Yes (uses $P^M(0,\cdot)$ and $f^M$) |
| Swaption pricing | Not standard — no clean affine formula for coupon bonds | Yes — Jamshidian decomposition |
| EMIR/IFRS 13 compliance | No — model price ≠ OIS price at $t=0$ | Yes — OIS-consistent by calibration |
| Computational cost | Low — exact simulation | Medium — Euler-Maruyama, $\theta(t)$ evaluation |

</small>


**When to use Vasicek:**
- Pedagogical benchmarks and convergence tests (QRE-48, QRE-49)
- Quick scenario analysis where exact OIS fit is not required
- When a parsimonious 3-parameter model is preferred for stability analysis

**When to use Hull-White:**
- Any XVA calculation (CVA, FVA, MVA) where exposure must be consistent with market pricing
- Swaption vol calibration and interest rate derivatives pricing
- Regulatory capital calculations under FRTB and SA-CCR
- Callable bond pricing under Longstaff-Schwartz (QRE-107) — HW provides the interest rate engine


In [ ]:
# ── Production class: HullWhiteProcess + MCSimulator ────────────────────────
from quant_risk.models.simulator import MCSimulator

# The curve is the only calibration input needed for θ(t)
# κ and σ come from the swaption calibration above
sim = MCSimulator(
    process   = HullWhiteProcess(curve=ois, kappa=kappa_cal, sigma=sigma_cal),
    x0        = r0,
    T         = 10.0,
    n_steps   = 120,
    n_paths   = 2000,
    antithetic= True,
    seed      = RNG_SEED,
)
print(sim.describe())

# ZCB prices via MCSimulator.sdf()
print("\nZCB prices via MCSimulator.sdf() vs OIS market")
print(f"  {'T':>5}  {'MC':>10}  {'OIS':>10}  {'Error (bps)':>12}")
for T_p in [1.0, 2.0, 5.0, 10.0]:
    P_mc  = sim.sdf(T_p).mean()
    P_ois = ois.discount(T_p)
    print(f"  {T_p:>5.1f}Y  {P_mc:>10.6f}  {P_ois:>10.6f}  {(P_mc-P_ois)*1e4:>+12.2f}")

# Exposure profile for a 5Y payer swap (receive fixed, pay floating)
# MTM = receiver swap value from t to swap maturity = sum of fixed coupons - floating leg
print("\nExposure profile skeleton (receiver swap: receive {r0:.2f}% fixed)")
swap_maturity = 5.0
swap_coupon   = r0       # ATM at inception (percent)
dates_exp = np.arange(0.5, sim.T + 0.5, 0.5)

def receiver_swap_mtm(paths: np.ndarray, t: float) -> np.ndarray:
    """
    Approximate receiver swap MTM at time t.
    Uses P^HW(t, Ti; r(t)) for each remaining payment date.
    """
    t_idx  = min(int(round(t * n_steps / T_sim)), n_steps)
    r_t    = paths[:, t_idx]
    remaining = np.arange(
        np.ceil(t * 2) / 2 + 0.5 * (t % 0.5 == 0),
        swap_maturity + 0.5, 0.5
    )
    remaining = remaining[remaining > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    yf = np.full(len(remaining), 0.5)
    P_ti  = hw_bond_price(t, remaining, r_t, kappa_cal, sigma_cal, ois)
    # Receiver: receive fixed coupon, pay floating (≈ par - last payment)
    fixed_leg    = (swap_coupon / 100 * 0.5 * P_ti).sum(axis=1)
    floating_leg = ois.discount(t) / ois.discount(t + 1e-6) if t > 1e-6 else 1.0
    # Simplified: MTM ≈ fixed annuity value - (1 - final ZCB)
    P_last = P_ti[:, -1]
    return fixed_leg - (ois.discount(t) - P_last * ois.discount(remaining[-1]))

profile = sim.exposure_profile(receiver_swap_mtm, dates_exp[:10])  # first 5Y
print(f"  {'Date':>6}  {'EE':>8}  {'NEE':>8}  {'PFE95':>8}")
for i, t in enumerate(profile['dates']):
    print(f"  {t:>6.1f}Y  {profile['EE'][i]:>8.4f}  {profile['NEE'][i]:>8.4f}  "
          f"{profile['PFE'][i]:>8.4f}")
print(f"  EPE: {profile['EPE']:.4f}")


---
## Summary
<small>

| Section | Key formula | Implementation |
|---|---|---|
| **HW SDE** | $dr = (\theta(t)-\kappa r)dt + \sigma dW$ | `HullWhiteProcess` (QRE-48 for full derivation) |
| **Future bond price** | $P^{HW}(t,T) = \frac{P^M(0,T)}{P^M(0,t)}\exp[-B(T-t)(r(t)-f^M(0,t)) - V(t,T)]$ | `hw_bond_price()` — used in swaption Jamshidian |
| **ZCB option** | $C = P^M(0,T_1)N(h_1) - X P^M(0,T_0)N(h_2)$ | `hw_zcb_call()` / `hw_zcb_put()` |
| **Jamshidian swaption** | $V_{\text{payer}} = \sum_i c_i \cdot \text{Put}(0,T_0,T_i,X^*_i)$ | `hw_jamshidian_payer()` |
| **ATM normal vol** | $\sigma_N = V / (A\sqrt{T_0/2\pi}) \times 10000$ | `bachelier_normal_vol()` |
| **κ–σ calibration** | $\min_{\kappa,\sigma}\sum_i w_i(\sigma_N^{\text{model}}-\sigma_N^{(i)})^2$ | Nelder-Mead multi-start |
| **No-arbitrage check** | $P^M(0,t)\cdot\mathbb{E}[P^{HW}(t,T)] = P^M(0,T)$ | Verified numerically |

</small>


**Next:** [QRE-45 — CIR Model](07_cir_model.ipynb) — the Cox-Ingersoll-Ross model, which enforces $r(t) \geq 0$ via a square-root diffusion and a non-central chi-squared transition density. CIR uses the `CIRProcess` class (QRE-46).
